In [27]:
import trimesh

mesh = trimesh.load("../data/test_house.obj", force="mesh")

print(type(mesh))
print("Граней:", len(mesh.faces))
print("Размеры (x, y, z):", mesh.extents)
print("Замкнутый ли меш:", mesh.is_watertight)

<class 'trimesh.base.Trimesh'>
Граней: 108
Размеры (x, y, z): [10.  15.   3.6]
Замкнутый ли меш: False


In [28]:
bbox = mesh.extents  # ширина, длина, высота охватывающей коробки

metrics = {
    "width": round(bbox[0], 2),
    "length": round(bbox[1], 2),
    "height": round(bbox[2], 2),
    "footprint_area": round(bbox[0] * bbox[1], 2),
    "bbox_volume": round(bbox[0] * bbox[1] * bbox[2], 2),
    "mesh_volume": round(mesh.volume, 2),
    "surface_area": round(mesh.area, 2),
    "height_to_width": round(bbox[2] / min(bbox[0], bbox[1]), 2),
}

for key, value in metrics.items():
    print(key, "=", value)

width = 10.0
length = 15.0
height = 3.6
footprint_area = 150.0
bbox_volume = 540.0
mesh_volume = 132.12
surface_area = 952.38
height_to_width = 0.36


In [29]:
normals = mesh.face_normals       # направление, куда смотрит каждая грань
areas = mesh.area_faces           # площадь каждой грани
centers = mesh.triangles_center   # центр каждой грани

def facade_info(axis, sign):
    # axis: 0 = x, 1 = y; sign: -1 = сторона к минимуму, +1 = к максимуму
    edge = mesh.bounds[0][axis] if sign < 0 else mesh.bounds[1][axis]
    looks_out = normals[:, axis] * sign > 0.9
    on_edge = abs(centers[:, axis] - edge) < 0.01
    actual = areas[looks_out & on_edge].sum()

    other = 0 if axis == 1 else 1
    full = mesh.extents[other] * mesh.extents[2]
    opening = full - actual
    return {
        "full_area": round(full, 2),
        "solid_area": round(actual, 2),
        "opening_area": round(opening, 2),
        "opening_ratio": round(opening / full, 2),
    }

facades = {
    "south": facade_info(1, -1),
    "north": facade_info(1, +1),
    "west": facade_info(0, -1),
    "east": facade_info(0, +1),
}

for side, info in facades.items():
    print(side, info)

metrics["facades"] = facades

south {'full_area': np.float64(36.0), 'solid_area': np.float64(30.0), 'opening_area': np.float64(6.0), 'opening_ratio': np.float64(0.17)}
north {'full_area': np.float64(36.0), 'solid_area': np.float64(36.0), 'opening_area': np.float64(0.0), 'opening_ratio': np.float64(0.0)}
west {'full_area': np.float64(54.0), 'solid_area': np.float64(54.0), 'opening_area': np.float64(0.0), 'opening_ratio': np.float64(0.0)}
east {'full_area': np.float64(54.0), 'solid_area': np.float64(54.0), 'opening_area': np.float64(0.0), 'opening_ratio': np.float64(0.0)}


In [30]:
import json

print(json.dumps(metrics, indent=2, ensure_ascii=False))
print("Предупреждения:")
for w_ in warnings:
    print(" -", w_)

{
  "width": 10.0,
  "length": 15.0,
  "height": 3.6,
  "footprint_area": 150.0,
  "bbox_volume": 540.0,
  "mesh_volume": 132.12,
  "surface_area": 952.38,
  "height_to_width": 0.36,
  "facades": {
    "south": {
      "full_area": 36.0,
      "solid_area": 30.0,
      "opening_area": 6.0,
      "opening_ratio": 0.17
    },
    "north": {
      "full_area": 36.0,
      "solid_area": 36.0,
      "opening_area": 0.0,
      "opening_ratio": 0.0
    },
    "west": {
      "full_area": 54.0,
      "solid_area": 54.0,
      "opening_area": 0.0,
      "opening_ratio": 0.0
    },
    "east": {
      "full_area": 54.0,
      "solid_area": 54.0,
      "opening_area": 0.0,
      "opening_ratio": 0.0
    }
  }
}
Предупреждения:
 - Меш не замкнут: объём и площадь поверхности приблизительны
 - Этажность оценена по высоте, а не измерена
 - Площадь пола оценена по габаритам: для сложной формы плана она завышена


In [42]:
def facade_info(mesh, axis, sign):
    edge = mesh.bounds[0][axis] if sign < 0 else mesh.bounds[1][axis]
    looks_out = mesh.face_normals[:, axis] * sign > 0.9
    on_edge = abs(mesh.triangles_center[:, axis] - edge) < 0.01
    actual = mesh.area_faces[looks_out & on_edge].sum()

    other = 0 if axis == 1 else 1
    full = mesh.extents[other] * mesh.extents[2]
    opening = full - actual
    return {
        "full_area": round(float(full), 2),
        "opening_area": round(float(opening), 2),
        "opening_ratio": round(float(opening / full), 2),
    }


def footprint_area(mesh, tolerance=0.05):
    z_min = mesh.bounds[0][2]
    looks_down = mesh.face_normals[:, 2] < -0.9
    at_bottom = abs(mesh.triangles_center[:, 2] - z_min) < tolerance
    return float(mesh.area_faces[looks_down & at_bottom].sum())


def analyze_mesh(path, floor_height=3.0):
    mesh = trimesh.load(path, force="mesh")
    w, l, h = (float(v) for v in mesh.extents)
    warnings = []

    if not mesh.is_watertight:
        warnings.append("Меш не замкнут: объём и площадь поверхности приблизительны")
    if max(w, l, h) > 200:
        warnings.append("Размеры очень большие: возможно, модель в миллиметрах, а не в метрах")
    if h > max(w, l) * 3:
        warnings.append("Модель сильно вытянута по высоте: возможно, вертикальная ось не та (Y вместо Z)")

    floors = max(1, round(h / floor_height))
    bbox_area = w * l
    footprint = footprint_area(mesh)
    if footprint < 0.5 * bbox_area:
        warnings.append("Площадь плана намного меньше габаритов: здание может стоять на опорах или иметь сложную форму")
    if footprint < 0.9 * bbox_area:
        warnings.append("План не прямоугольный: «проёмы» на фасадах могут быть уступами здания, а не окнами")
    warnings.append("Этажность оценена по высоте, а не измерена")
    warnings.append("Площадь пола = площадь плана x этажность: для зданий с разной формой этажей это приблизительно")

    metrics = {
        "width_m": round(w, 2),
        "length_m": round(l, 2),
        "height_m": round(h, 2),
        "footprint_area_m2": round(footprint, 2),
        "bbox_fill_ratio": round(footprint / bbox_area, 2),
        "floors_estimate": floors,
        "floor_area_estimate_m2": round(footprint * floors, 2),
        "mesh_volume_m3": round(float(mesh.volume), 2),
        "height_to_width": round(h / min(w, l), 2),
        "facades": {
            "south": facade_info(mesh, 1, -1),
            "north": facade_info(mesh, 1, +1),
            "west": facade_info(mesh, 0, -1),
            "east": facade_info(mesh, 0, +1),
        },
        "watertight": bool(mesh.is_watertight),
    }
    return metrics, warnings

In [32]:
for ext in ["obj", "stl", "glb"]:
    m, w_ = analyze_mesh(f"../data/test_house.{ext}")
    print(ext, m["width_m"], m["length_m"], m["height_m"], m["facades"]["south"]["opening_area"])

obj 10.0 15.0 3.6 6.0
stl 10.0 15.0 3.6 6.0
glb 10.0 15.0 3.6 6.0


In [33]:
def make_report(metrics, warnings):
    pros = []
    cons = []

    # пропорции
    hw = metrics["height_to_width"]
    if hw < 0.5:
        pros.append(f"Низкий, приземистый объём (высота/ширина = {hw})")
    elif hw > 1.5:
        cons.append(f"Вытянутый вверх объём (высота/ширина = {hw}): может требовать усиленной конструкции")

    # остекление южного фасада
    south = metrics["facades"]["south"]
    ratio = south["opening_ratio"]
    if ratio == 0:
        cons.append("На южном фасаде нет проёмов: дневной свет и солнечные приобретения ограничены")
    elif ratio < 0.15:
        cons.append(f"Остекление юга небольшое ({ratio:.0%}): возможна нехватка дневного света")
    elif ratio <= 0.4:
        pros.append(f"Разумная доля остекления юга ({ratio:.0%}): баланс света и теплопотерь")
    else:
        cons.append(f"Очень большое остекление юга ({ratio:.0%}): риск перегрева летом")

    # остальные фасады
    for side, name in [("north", "северном"), ("west", "западном"), ("east", "восточном")]:
        if metrics["facades"][side]["opening_ratio"] == 0:
            cons.append(f"На {name} фасаде нет проёмов")

    print("ПЛЮСЫ:")
    for p in pros:
        print(" +", p)
    print("МИНУСЫ:")
    for c in cons:
        print(" -", c)
    print("ОГРАНИЧЕНИЯ ТОЧНОСТИ:")
    for w in warnings:
        print(" !", w)


metrics, warnings = analyze_mesh("../data/test_house.obj")
make_report(metrics, warnings)

ПЛЮСЫ:
 + Низкий, приземистый объём (высота/ширина = 0.36)
 + Разумная доля остекления юга (17%): баланс света и теплопотерь
МИНУСЫ:
 - На северном фасаде нет проёмов
 - На западном фасаде нет проёмов
 - На восточном фасаде нет проёмов
ОГРАНИЧЕНИЯ ТОЧНОСТИ:
 ! Меш не замкнут: объём и площадь поверхности приблизительны
 ! Этажность оценена по высоте, а не измерена
 ! Площадь пола = площадь плана x этажность: для зданий с разной формой этажей это приблизительно


In [34]:
import trimesh
import numpy as np

# 1. высокая башня 8 x 8 x 24 м (примерно 8 этажей)
tower = trimesh.creation.box(extents=[8, 8, 24])
tower.apply_translation([4, 4, 12])
tower.export("../data/test_tower.obj")

# 2. Г-образный дом (две коробки, склеенные вместе)
part_a = trimesh.creation.box(extents=[12, 5, 3])
part_a.apply_translation([6, 2.5, 1.5])
part_b = trimesh.creation.box(extents=[5, 10, 3])
part_b.apply_translation([2.5, 10, 1.5])
l_shape = trimesh.util.concatenate([part_a, part_b])
l_shape.export("../data/test_lshape.obj")

# 3. модель в миллиметрах (как будто автор экспортировал не в метрах)
mm_house = trimesh.creation.box(extents=[10000, 15000, 3000])
mm_house.export("../data/test_mm.obj")

print("Три файла созданы")

Три файла созданы


In [35]:
for name in ["tower", "lshape", "mm"]:
    m, w_ = analyze_mesh(f"../data/test_{name}.obj")
    print("=====", name, "=====")
    print("габариты:", m["width_m"], m["length_m"], m["height_m"])
    print("этажей:", m["floors_estimate"], "| пол (оценка):", m["floor_area_estimate_m2"], "м²")
    print("пропорции:", m["height_to_width"])
    print("замкнут:", m["watertight"])
    for x in w_:
        print("  !", x)

===== tower =====
габариты: 8.0 8.0 24.0
этажей: 8 | пол (оценка): 512.0 м²
пропорции: 3.0
замкнут: True
  ! Этажность оценена по высоте, а не измерена
  ! Площадь пола = площадь плана x этажность: для зданий с разной формой этажей это приблизительно
===== lshape =====
габариты: 12.0 15.0 3.0
этажей: 1 | пол (оценка): 110.0 м²
пропорции: 0.25
замкнут: False
  ! Меш не замкнут: объём и площадь поверхности приблизительны
  ! Этажность оценена по высоте, а не измерена
  ! Площадь пола = площадь плана x этажность: для зданий с разной формой этажей это приблизительно
===== mm =====
габариты: 10000.0 15000.0 3000.0
этажей: 1000 | пол (оценка): 150000000000.0 м²
пропорции: 0.3
замкнут: True
  ! Размеры очень большие: возможно, модель в миллиметрах, а не в метрах
  ! Этажность оценена по высоте, а не измерена
  ! Площадь пола = площадь плана x этажность: для зданий с разной формой этажей это приблизительно


In [36]:
def footprint_area(mesh, tolerance=0.05):
    z_min = mesh.bounds[0][2]
    looks_down = mesh.face_normals[:, 2] < -0.9
    at_bottom = abs(mesh.triangles_center[:, 2] - z_min) < tolerance
    return float(mesh.area_faces[looks_down & at_bottom].sum())


for name in ["house", "tower", "lshape"]:
    m = trimesh.load(f"../data/test_{name}.obj", force="mesh")
    real = footprint_area(m)
    by_bbox = m.extents[0] * m.extents[1]
    print(name, "| по габаритам:", round(by_bbox, 2), "| по нижним граням:", round(real, 2))

house | по габаритам: 150.0 | по нижним граням: 150.0
tower | по габаритам: 64.0 | по нижним граням: 64.0
lshape | по габаритам: 180.0 | по нижним граням: 110.0


In [37]:
for name in ["house", "tower", "lshape"]:
    m, w_ = analyze_mesh(f"../data/test_{name}.obj")
    print(name, "| план:", m["footprint_area_m2"], "| заполненность:", m["bbox_fill_ratio"],
          "| этажей:", m["floors_estimate"], "| пол:", m["floor_area_estimate_m2"])

house | план: 150.0 | заполненность: 1.0 | этажей: 1 | пол: 150.0
tower | план: 64.0 | заполненность: 1.0 | этажей: 8 | пол: 512.0
lshape | план: 110.0 | заполненность: 0.61 | этажей: 1 | пол: 110.0


In [38]:
import plotly.graph_objects as go

def show_mesh(mesh):
    v, f = mesh.vertices, mesh.faces
    fig = go.Figure(data=[go.Mesh3d(
        x=v[:, 0], y=v[:, 1], z=v[:, 2],
        i=f[:, 0], j=f[:, 1], k=f[:, 2],
        color="lightgray", flatshading=True,
    )])
    fig.update_layout(scene=dict(aspectmode="data"), margin=dict(l=0, r=0, t=0, b=0))
    fig.show()

path = "../data/test_lshape.obj"   # замените на имя вашего файла
show_mesh(trimesh.load(path, force="mesh"))

In [43]:
metrics, warnings = analyze_mesh(path)
print(json.dumps(metrics, indent=2, ensure_ascii=False))
make_report(metrics, warnings)

{
  "width_m": 12.0,
  "length_m": 15.0,
  "height_m": 3.0,
  "footprint_area_m2": 110.0,
  "bbox_fill_ratio": 0.61,
  "floors_estimate": 1,
  "floor_area_estimate_m2": 110.0,
  "mesh_volume_m3": 330.0,
  "height_to_width": 0.25,
  "facades": {
    "south": {
      "full_area": 36.0,
      "opening_area": 0.0,
      "opening_ratio": 0.0
    },
    "north": {
      "full_area": 36.0,
      "opening_area": 21.0,
      "opening_ratio": 0.58
    },
    "west": {
      "full_area": 45.0,
      "opening_area": 0.0,
      "opening_ratio": 0.0
    },
    "east": {
      "full_area": 45.0,
      "opening_area": 30.0,
      "opening_ratio": 0.67
    }
  },
  "watertight": false
}
ПЛЮСЫ:
 + Низкий, приземистый объём (высота/ширина = 0.25)
МИНУСЫ:
 - На южном фасаде нет проёмов: дневной свет и солнечные приобретения ограничены
 - На западном фасаде нет проёмов
ОГРАНИЧЕНИЯ ТОЧНОСТИ:
 ! Меш не замкнут: объём и площадь поверхности приблизительны
 ! План не прямоугольный: «проёмы» на фасадах могут быт

In [44]:
path = "../data/test_lshape.obj"
metrics, warnings = analyze_mesh(path)
for w_ in warnings:
    print(" !", w_)

 ! Меш не замкнут: объём и площадь поверхности приблизительны
 ! План не прямоугольный: «проёмы» на фасадах могут быть уступами здания, а не окнами
 ! Этажность оценена по высоте, а не измерена
 ! Площадь пола = площадь плана x этажность: для зданий с разной формой этажей это приблизительно


In [45]:
def build_prompt(metrics, warnings):
    rules = """Ты помощник архитектора. Ниже измерения 3D-модели здания и предупреждения о их точности.

ПРАВИЛА:
1. Используй ТОЛЬКО числа из блока ИЗМЕРЕНИЯ. Не придумывай новых чисел.
2. Каждый вывод должен ссылаться на конкретное измерение.
3. Если вывод зависит от неточной метрики (см. ПРЕДУПРЕЖДЕНИЯ), прямо напиши, что он приблизителен.
4. Ничего не утверждай про то, чего нет в измерениях (конструкцию, материалы, инсоляцию, нормативы).
5. Не называй проёмы окнами, если предупреждение говорит, что это могут быть уступы здания.
6. Верни ТОЛЬКО JSON такого вида:
{"pros": [{"text": "...", "based_on": "название метрики"}],
 "cons": [{"text": "...", "based_on": "название метрики"}],
 "caveats": ["..."]}
"""
    data = json.dumps(metrics, indent=2, ensure_ascii=False)
    warn = "\n".join("- " + w for w in warnings)
    return f"{rules}\nИЗМЕРЕНИЯ:\n{data}\n\nПРЕДУПРЕЖДЕНИЯ:\n{warn}\n"


path = "../data/test_lshape.obj"
metrics, warnings = analyze_mesh(path)
prompt = build_prompt(metrics, warnings)
print(prompt)

Ты помощник архитектора. Ниже измерения 3D-модели здания и предупреждения о их точности.

ПРАВИЛА:
1. Используй ТОЛЬКО числа из блока ИЗМЕРЕНИЯ. Не придумывай новых чисел.
2. Каждый вывод должен ссылаться на конкретное измерение.
3. Если вывод зависит от неточной метрики (см. ПРЕДУПРЕЖДЕНИЯ), прямо напиши, что он приблизителен.
4. Ничего не утверждай про то, чего нет в измерениях (конструкцию, материалы, инсоляцию, нормативы).
5. Не называй проёмы окнами, если предупреждение говорит, что это могут быть уступы здания.
6. Верни ТОЛЬКО JSON такого вида:
{"pros": [{"text": "...", "based_on": "название метрики"}],
 "cons": [{"text": "...", "based_on": "название метрики"}],
 "caveats": ["..."]}

ИЗМЕРЕНИЯ:
{
  "width_m": 12.0,
  "length_m": 15.0,
  "height_m": 3.0,
  "footprint_area_m2": 110.0,
  "bbox_fill_ratio": 0.61,
  "floors_estimate": 1,
  "floor_area_estimate_m2": 110.0,
  "mesh_volume_m3": 330.0,
  "height_to_width": 0.25,
  "facades": {
    "south": {
      "full_area": 36.0,
     

In [46]:
import re

def collect_metric_names(d, names=None):
    # собирает все названия метрик, включая вложенные (facades -> south -> ...)
    if names is None:
        names = set()
    for key, value in d.items():
        names.add(key)
        if isinstance(value, dict):
            collect_metric_names(value, names)
    return names


def collect_numbers(d, numbers=None):
    # собирает все числа из измерений
    if numbers is None:
        numbers = set()
    for value in d.values():
        if isinstance(value, dict):
            collect_numbers(value, numbers)
        elif isinstance(value, (int, float)) and not isinstance(value, bool):
            numbers.add(round(float(value), 2))
    return numbers


def check_report(report, metrics):
    problems = []
    names = collect_metric_names(metrics)
    known = collect_numbers(metrics)

    for kind in ["pros", "cons"]:
        for item in report.get(kind, []):
            if item.get("based_on") not in names:
                problems.append(f"Нет такой метрики: {item.get('based_on')!r}")
            for num in re.findall(r"\d+(?:[.,]\d+)?", item.get("text", "")):
                value = round(float(num.replace(",", ".")), 2)
                if value not in known:
                    problems.append(f"Число {num} не из измерений: {item['text']!r}")
    return problems

In [47]:
fake_report = {
    "pros": [
        {"text": "Низкий объём: высота к ширине 0.25", "based_on": "height_to_width"},
        {"text": "Площадь плана 110.0 м², этаж один", "based_on": "footprint_area_m2"},
    ],
    "cons": [
        {"text": "Заполненность габаритов 0.61, план сложной формы", "based_on": "bbox_fill_ratio"},
        {"text": "Низкая энергоэффективность, потери 85 кВт", "based_on": "energy_loss"},
    ],
    "caveats": ["Этажность оценена по высоте"],
}

for p in check_report(fake_report, metrics):
    print(" -", p)

 - Нет такой метрики: 'energy_loss'
 - Число 85 не из измерений: 'Низкая энергоэффективность, потери 85 кВт'
